# AdamWorld — generate a 3D world on a free Colab GPU

Runs the default pipeline on a Colab **T4 (~16 GB)**. Set *Runtime → Change runtime
type → T4 GPU* first. The stage cache lives under `./world/cache`, so re-running
resumes after a disconnect.

In [ ]:
import os, subprocess, sys

# Provide the adamworld source: upload the repo's core/ folder, or set GIT_URL to clone.
GIT_URL = ""   # e.g. "https://github.com/<you>/adamworld.git"
SRC = "adamworld/core" if os.path.isdir("adamworld/core") else "core"
if not os.path.isdir(SRC) and GIT_URL:
    subprocess.run(["git", "clone", "--depth", "1", GIT_URL, "adamworld"], check=True)
    SRC = "adamworld/core"
assert os.path.isdir(SRC), "Upload core/ (File browser) or set GIT_URL to clone the repo."

subprocess.run([sys.executable, "-m", "pip", "install", "-q", SRC], check=True)
subprocess.run([sys.executable, "-m", "pip", "install", "-q",
                "diffusers==0.31.0", "transformers==4.46.2", "accelerate==1.1.1",
                "safetensors", "huggingface_hub", "opencv-python-headless", "gsplat"], check=True)
print("install done")

In [ ]:
import os, torch
os.environ["ADAMWORLD_CACHE"] = os.path.abspath("./.adamworld_cache")
print("CUDA:", torch.cuda.is_available(), torch.cuda.get_device_name(0) if torch.cuda.is_available() else "-")

In [ ]:
# --- parameters ---
PROMPT     = "a quiet forest clearing with a mossy stone well, soft morning light"
IMAGE      = None          # or "/kaggle/input/.../photo.jpg" to start from an image
RESOLUTION = 1024          # lower to 512/768 if you hit OOM
BACKEND    = "default"     # "default" (SDXL/FLUX/DepthAnything) | "flux-pano" (NC) | "mock"
SEED       = 0
OFFLOAD    = True          # sequential CPU offload — required to fit FLUX/SDXL on 16 GB

In [ ]:
from pathlib import Path
from adamworld.pipeline import Pipeline
from adamworld.profiling import Profiler
from adamworld.config import Settings
from adamworld.types import GenerationRequest

settings = Settings(offload=OFFLOAD)
prof = Profiler()
def progress(stage, frac):
    print(f"{frac*100:6.1f}%  {stage}", flush=True)

req = GenerationRequest(
    prompt=PROMPT,
    image_paths=[Path(IMAGE)] if IMAGE else [],
    resolution=RESOLUTION,
    backend=BACKEND,
    seed=SEED,
)
world = Pipeline(settings=settings, profiler=prof, progress=progress).generate(
    req, Path("./world")
)
print("\n", prof.table(), sep="")
print("world:", world.root)

In [ ]:
from IPython.display import Image as IPyImage, display
from google.colab import files
import shutil
display(IPyImage(filename=str(world.thumbnail)))
shutil.make_archive("world", "zip", world.root)
files.download("world.zip")